# Split Datasets For all stages

Split dataset to Train/Validation/Test sets for all three stages

## A. Overview

- Call (B)inary is all tweet that only has Informative label and missing Humnaitarian
- Call (H)umanitarian is all tweet that has Humnaitarian, means Informative label is `True`

### A.1. Stages:

#### 1. BERT
- Binary classification
- Only need `informative` label with True/False values

#### 2. Llama
- Binary classification
- Only need `informative` label with True/False values
- Selected from False positive record of stage 1 only.

#### 3. Llama
- Multi-label classification
- Need `humanitarian` labels

### A.2. Spliting steps:

#### 1. Global Test sets
- Stratified to from base data from both (B) and (H)
- Random split
- 10%
- Isolate this part, do not use in any trainng step, only use for final test for all three stages

#### 2. Stage 1 - Crossed strategy:
- Stratified to from base data from both (B) and (H)
- Random split
- 10-20% of remain sets
- Ensuring ~10_000 training records (which considered large size)

#### 3. Stage 2 - Crossed strategy:
- Llama prefer quality than quantity.
- 5,000 to 10,000 high-quality is considered good size.
- run all through BERT and get wrong predictiong as training input

#### 4. Stage 1 - Isolate strategy:
- all record that do not have `humanitarian` labels

#### 5. Stage 2 - Isolate strategy:
- Llama prefer quality than quantity
- `humanitarian` record only
- run all through BERT and get wrong predict and low confidence score record to train

#### 6. Stage 3:
- Use all (H) set.
- The Safety Net
  - Rerun the False record in (B) throught both stage 1 and 2
  - Isolate the False Positives (can fool both pre-steps)
  - Label them as `non-humanitarian` to reduce the model forcing to specified humanitarian category.

## B. Split STAGEs Datasets

In [1]:
import math
import csv
import os
import random
from pathlib import Path

from sklearn.model_selection import train_test_split
import pandas as pd
import matplotlib.pyplot as plt

import configuration
from src import data_utils, setup

random.seed(setup.RANDOM_SEED)

In [2]:
dataset_path = Path("..") / "data"
disaster_path = dataset_path / "disaster"
extended_filepath = dataset_path / "extended"
splitted_filepath = dataset_path / "splitted"

out_topic_times = 20

### B.1. Load Original Datasets

In [3]:
df_disaster_original = pd.read_csv(
    dataset_path / "disaster" / data_utils.DISASTER_FILE
).sample(frac=1, random_state=setup.RANDOM_SEED)
df_weather_original = pd.read_csv(extended_filepath / data_utils.WEATHER_FILE).sample(
    frac=1, random_state=setup.RANDOM_SEED
)
df_out_topic_original = pd.read_csv(
    extended_filepath / data_utils.OUT_TOPIC_FILE
).sample(frac=1, random_state=setup.RANDOM_SEED)

df_humanitarian_original = df_disaster_original[
    ~df_disaster_original["humanitarian_label"].isna()
]

In [4]:
print(
    f"Base disaster dataset size: {df_disaster_original.groupby('informative').size()}"
)
print(
    f"Base humanitarian dataset size: {df_humanitarian_original.groupby('informative').size()}"
)
print(f"Base weather dataset size: {len(df_weather_original)}")
print(f"Base out-topic dataset size: {len(df_out_topic_original)}")

Base disaster dataset size: informative
False    60926
True     98804
dtype: int64
Base humanitarian dataset size: informative
False    27769
True     65733
dtype: int64
Base weather dataset size: 28643
Base out-topic dataset size: 3577546


In [5]:
def n_out_topic_samples(df_disaster, df_weather, times):    
    n_disaster_true = len(df_disaster[df_disaster["informative"] == True])
    n_disaster_false = len(df_disaster) - n_disaster_true
    # ceiling to ensure the fraction is not smaller than the specified fraction due to rounding
    n_total_false = math.ceil(n_disaster_true * times)
    return n_total_false - n_disaster_false - len(df_weather)
    
    return n_out_topic
def split_fraction(
    df_disaster_original,
    df_weather_original,
    df_out_topic_original,
    fraction=0.1,
    out_topic_times=20,
    random_state=setup.RANDOM_SEED,
    file_path: Path = None,
):
    df_disaster = df_disaster_original.sample(frac=fraction, random_state=random_state)
    df_weather = df_weather_original.sample(frac=fraction, random_state=random_state)
    
    n_out_topic = n_out_topic_samples(df_disaster, df_weather, out_topic_times)

    df = pd.concat(
        [
            df_disaster,
            df_weather,
            df_out_topic_original.sample(n=n_out_topic, random_state=random_state),
        ],
        ignore_index=True,
    )

    print(f"Fraction set size: {len(df)}")
    print(
        f"Disaster informative samples: {df[df['subset'] == 'disaster'].groupby('informative').size()}"
    )
    print(
        f"Humanitarian samples: {df[(df['subset'] == 'disaster') & (df['humanitarian_label'].notnull())].groupby(['informative']).size()}"
    )
    print(
        f"Humanitarian sub-label samples: {df[df['subset'] == 'disaster'].groupby(['humanitarian_label', 'informative']).size()}"
    )
    print(f"Fraction set weather samples: {len(df[df['subset'] == 'weather'])}")
    print(f"Fraction set out-topic samples: {len(df[df['subset'] == 'out_topic'])}")

    display(df.head())

    # Remove the samples from the original datasets
    df_uids = set(df["uid"])
    df_disaster_remaining = df_disaster_original[
        ~df_disaster_original["uid"].isin(df_uids)
    ]
    df_weather_remaining = df_weather_original[
        ~df_weather_original["uid"].isin(df_uids)
    ]
    df_out_topic_remaining = df_out_topic_original[
        ~df_out_topic_original["uid"].isin(df_uids)
    ]

    file_path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(
        file_path,
        index=False,
        quoting=csv.QUOTE_NONNUMERIC,
    )
    return df, df_disaster_remaining, df_weather_remaining, df_out_topic_remaining

### B.2. Global Test set

In [6]:
test_frac = 0.1
test_path = splitted_filepath / "global_test_sets.csv"
print("Splitting the Global TEST datasets...")

df_test, df_disaster, df_weather, df_out_topic = split_fraction(
    df_disaster_original=df_disaster_original,
    df_weather_original=df_weather_original,
    df_out_topic_original=df_out_topic_original,
    fraction=test_frac,
    out_topic_times=out_topic_times,
    random_state=setup.RANDOM_SEED,
    file_path=test_path,
)

Splitting the Global TEST datasets...
Fraction set size: 206703
Disaster informative samples: informative
False    6130
True     9843
dtype: int64
Humanitarian samples: informative
False    2747
True     6566
dtype: int64
Humanitarian sub-label samples: humanitarian_label                      informative
affected_individuals                    False            13
                                        True             32
displaced_people_and_evacuations        True            363
infrastructure_and_utility_damage       False            39
                                        True            722
injured_or_dead_people                  False            24
                                        True            432
missing_trapped_or_found_people         False             2
                                        True             35
not_humanitarian                        False          1039
                                        True            457
not_related_or_irrelevant         

Fraction set out-topic samples: 187866


,uid,tweet_text,informative,humanitarian_label,subset
0,208755,"I live with six persons, we would like to find...",True,affected_individuals,disaster
1,13936,The final seconds of Pierson’s 1-0 win in the ...,True,NaN,disaster
2,226927,@mansionz I am standing on the beach watching ...,True,other_relevant_information,disaster
3,78148,Video: Market View: 'Frankenstorm' shutters U....,True,NaN,disaster
4,183211,2.1 Due to sporadic skirmishes in eastern D.R....,True,not_humanitarian,disaster


## C. CROSS Strategy
### C.1. STAGE 1 - BERT

In [7]:
bert_frac = 0.1
bert_path = splitted_filepath / "stage_1" / "crossed" / "bert_sets.csv"
print("Splitting the CROSSED BERT datasets...")

(
    df_crossed_bert,
    df_crossed_disaster_llama,
    df_crossed_weather_llama,
    df_crossed_out_topic_llama,
) = split_fraction(
    df_disaster_original=df_disaster,
    df_weather_original=df_weather,
    df_out_topic_original=df_out_topic,
    fraction=bert_frac,
    out_topic_times=out_topic_times,
    random_state=setup.RANDOM_SEED,
    file_path=bert_path,
)

Splitting the CROSSED BERT datasets...
Fraction set size: 186606
Disaster informative samples: informative
False    5490
True     8886
dtype: int64
Humanitarian samples: informative
False    2524
True     5943
dtype: int64
Humanitarian sub-label samples: humanitarian_label                      informative
affected_individuals                    False            13
                                        True             26
displaced_people_and_evacuations        True            293
infrastructure_and_utility_damage       False            49
                                        True            669
injured_or_dead_people                  False            19
                                        True            362
missing_trapped_or_found_people         False             3
                                        True             43
not_humanitarian                        False           931
                                        True            433
not_related_or_irrelevant        

,uid,tweet_text,informative,humanitarian_label,subset
0,221446,Some humanitarian supplies have reportedly bee...,True,other_relevant_information,disaster
1,187547,PHOTOS: Hurricane Irma damage across Middle Ge...,True,infrastructure_and_utility_damage,disaster
2,194604,"Me and Hadi never sleep, loool\n 1:12 &amp; I ...",False,NaN,disaster
3,99917,@Loopermovie Wow. Amazing. I cried at the end....,False,NaN,disaster
4,101962,"2e Atelier v Selfcity - met Repair Cafφ⌐, Peer...",False,not_related_or_irrelevant,disaster


### C.2. STAGE 2 - LLAMA

In [8]:
# put all the remaining samples into the LLAMA set
df_crossed_llama = pd.concat(
    [df_crossed_disaster_llama, df_crossed_weather_llama, df_crossed_out_topic_llama],
    ignore_index=True,
)

print(f"CROSSED LLAMA set size: {len(df_crossed_llama)}")
print(
    f"Disaster informative samples: {df_crossed_llama[df_crossed_llama['subset'] == 'disaster'].groupby('informative').size()}"
)
print(
    f"Humanitarian samples: {df_crossed_llama[(df_crossed_llama['subset'] == 'disaster') & (df_crossed_llama['humanitarian_label'].notnull())].groupby(['informative']).size()}"
)
print(
    f"Humanitarian_label samples: {df_crossed_llama[df_crossed_llama['subset'] == 'disaster'].groupby(['humanitarian_label', 'informative']).size()}"
)
print(
    f"LLAMA set weather samples: {len(df_crossed_llama[df_crossed_llama['subset'] == 'weather'])}"
)
print(
    f"LLAMA set out-topic samples: {len(df_crossed_llama[df_crossed_llama['subset'] == 'out_topic'])}"
)
df_crossed_llama_path = splitted_filepath / "stage_2" / "crossed" / "llama_pre_bert_sets.csv"
df_crossed_llama_path.parent.mkdir(parents=True, exist_ok=True)
df_crossed_llama.to_csv(
    df_crossed_llama_path,
    index=False,
    quoting=csv.QUOTE_NONNUMERIC,
)

CROSSED LLAMA set size: 3372610
Disaster informative samples: informative
False    49306
True     80075
dtype: int64
Humanitarian samples: informative
False    22498
True     53224
dtype: int64
Humanitarian_label samples: humanitarian_label                      informative
affected_individuals                    False            126
                                        True             257
displaced_people_and_evacuations        True            2626
infrastructure_and_utility_damage       False            360
                                        True            5850
injured_or_dead_people                  False            165
                                        True            3500
missing_trapped_or_found_people         False             15
                                        True             392
not_humanitarian                        False           8452
                                        True            3934
not_related_or_irrelevant               False          

## D. ISOLATED Strategy
### D.1. STAGE 1 - BERT

In [9]:
# Only keep the samples that are not in the humanitarian dataset for the BERT dataset
df_iso_bert_disaster = df_disaster[df_disaster["humanitarian_label"].isna()]
df_iso_bert_weather = df_weather.sample(frac=0.5, random_state=setup.RANDOM_SEED)

n_iso_bert_out_topic = n_out_topic_samples(
    df_iso_bert_disaster, df_iso_bert_weather, out_topic_times
)

df_iso_bert_out_topic = df_out_topic.sample(
    n=n_iso_bert_out_topic, random_state=setup.RANDOM_SEED
)

# Remove the samples from the original datasets
df_iso_lla_disaster = df_disaster[~df_disaster["uid"].isin(df_iso_bert_disaster["uid"])]
df_iso_lla_weather = df_weather[~df_weather["uid"].isin(df_iso_bert_weather["uid"])]
df_iso_lla_out_topic = df_out_topic[
    ~df_out_topic["uid"].isin(df_iso_bert_out_topic["uid"])
]

df_iso_bert = pd.concat(
    [df_iso_bert_disaster, df_iso_bert_weather, df_iso_bert_out_topic],
    ignore_index=True,
)

df_iso_bert_path = splitted_filepath / "stage_1" / "isolated" / "bert_sets.csv"
df_iso_bert_path.parent.mkdir(parents=True, exist_ok=True)
df_iso_bert.to_csv(
    df_iso_bert_path,
    index=False,
    quoting=csv.QUOTE_NONNUMERIC,
)

print(f"ISOLATED BERT set size: {len(df_iso_bert)}")
print(
    f"Disaster informative samples: {df_iso_bert[df_iso_bert['subset'] == 'disaster'].groupby('informative').size()}"
)
print(
    f"ISOLATED BERT set weather samples: {len(df_iso_bert[df_iso_bert['subset'] == 'weather'])}"
)
print(
    f"ISOLATED BERT set out-topic samples: {len(df_iso_bert[df_iso_bert['subset'] == 'out_topic'])}"
)

ISOLATED BERT set size: 625674
Disaster informative samples: informative
False    29774
True     29794
dtype: int64
ISOLATED BERT set weather samples: 12890
ISOLATED BERT set out-topic samples: 553216


### D.2. STAGE 2 - Llama

In [10]:
# put all the remaining samples into the LLAMA set
df_iso_llama = pd.concat(
    [df_iso_lla_disaster, df_iso_lla_weather, df_iso_lla_out_topic],
    ignore_index=True,
)

print(f"LLAMA set size: {len(df_iso_llama)}")
print(
    f"Disaster informative samples: {df_iso_llama[df_iso_llama['subset'] == 'disaster'].groupby('informative').size()}"
)
print(
        f"Humanitarian samples: {df_iso_llama[(df_iso_llama['subset'] == 'disaster') & (df_iso_llama['humanitarian_label'].notnull())].groupby(['informative']).size()}"
    )
print(
    f"Humanitarian_label samples: {df_iso_llama[df_iso_llama['subset'] == 'disaster'].groupby(['humanitarian_label', 'informative']).size()}"
)
print(f"LLAMA set weather samples: {len(df_iso_llama[df_iso_llama['subset'] == 'weather'])}")
print(
    f"LLAMA set out-topic samples: {len(df_iso_llama[df_iso_llama['subset'] == 'out_topic'])}"
)

df_iso_llama_path = splitted_filepath / "stage_2" / "isolated" / "llama_pre_bert_sets.csv"
df_iso_llama_path.parent.mkdir(parents=True, exist_ok=True)
df_iso_llama.to_csv(
    splitted_filepath / "stage_2" / "isolated" / "llama_pre_bert_sets.csv",
    index=False,
    quoting=csv.QUOTE_NONNUMERIC,
)

LLAMA set size: 2933542


Disaster informative samples: informative
False    25022
True     59167
dtype: int64
Humanitarian samples: informative
False    25022
True     59167
dtype: int64
Humanitarian_label samples: humanitarian_label                      informative
affected_individuals                    False            139
                                        True             283
displaced_people_and_evacuations        True            2919
infrastructure_and_utility_damage       False            409
                                        True            6519
injured_or_dead_people                  False            184
                                        True            3862
missing_trapped_or_found_people         False             18
                                        True             435
not_humanitarian                        False           9383
                                        True            4367
not_related_or_irrelevant               False          12236
other_relevant_information

## E. Stage 3 - Multi label LLama

In [11]:
df_humanitarian = df_disaster[~df_disaster["humanitarian_label"].isna()]
df_humanitarian_path = splitted_filepath / "stage_3" / "humanitarian.csv"
df_humanitarian_path.parent.mkdir(parents=True, exist_ok=True)
df_humanitarian.to_csv(
    df_humanitarian_path,
    index=False,
    quoting=csv.QUOTE_NONNUMERIC,
)
print("Humanitarian set:")
print(df_humanitarian.groupby("informative").size())


df_no_informative = df_disaster[df_disaster["informative"] == False]
df_no_informative_path = splitted_filepath / "stage_3" / "no_informative.csv"
df_no_informative_path.parent.mkdir(parents=True, exist_ok=True)
df_no_informative.to_csv(
    df_no_informative_path,
    index=False,
    quoting=csv.QUOTE_NONNUMERIC,
)
print("No informative set:")
print(df_no_informative.groupby("informative").size())

Humanitarian set:
informative
False    25022
True     59167
dtype: int64
No informative set:
informative
False    54796
dtype: int64
